In [1]:
# Cell 0: Imports and basic config

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from typing import Dict, List, Tuple

torch.set_printoptions(precision=4, sci_mode=False)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMB_PATH = "data/dolma_300_2024_1.2M.100_combined.txt"  # or GloVe path
EMB_DIM = 300


In [2]:
# Cell 1: Load embeddings (text format: word dim1 ... dimD)

def load_embeddings(path: str) -> Dict[str, np.ndarray]:
    """
    Load word embeddings from a plain text file.

    Expected format:
        token dim1 dim2 ... dimD
    """
    embs: Dict[str, np.ndarray] = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 10:
                continue
            word = parts[0]
            vec = np.asarray(parts[1:], dtype=np.float32)
            embs[word] = vec
    return embs


def get_vec(word: str, embs: Dict[str, np.ndarray]) -> torch.Tensor:
    """Return embedding tensor for a given word."""
    if word not in embs:
        raise ValueError(f"Word '{word}' not found in embeddings.")
    return torch.tensor(embs[word], dtype=torch.float32, device=DEVICE)


print("Loading embeddings...")
embs = load_embeddings(EMB_PATH)
print("Loaded", len(embs), "tokens.")
EMB_DIM = len(next(iter(embs.values())))
print("Embedding dim:", EMB_DIM)


Loading embeddings...
Loaded 1200001 tokens.
Embedding dim: 300


In [3]:
# Cell 2: Hand-crafted triplet list (child1, child2, parent)

TRIPLETS: List[Tuple[str, str, str]] = [
    ("cat", "dog", "animal"),
    ("tiger", "lion", "animal"),
    ("apple", "banana", "fruit"),
    ("car", "bus", "vehicle"),
    ("piano", "guitar", "instrument"),
    ("happy", "sad", "emotion"),
    ("red", "blue", "color"),
    ("king", "queen", "royalty"),
    ("seoul", "tokyo", "city"),
]

# Filter out missing words
valid_triplets: List[Tuple[str, str, str]] = []
missing = set()

for w1, w2, wp in TRIPLETS:
    if w1 not in embs or w2 not in embs or wp not in embs:
        missing.update([w for w in [w1, w2, wp] if w not in embs])
        continue
    valid_triplets.append((w1, w2, wp))

print("Valid triplets:", len(valid_triplets))
if missing:
    print("Missing tokens (skipped):", missing)


Valid triplets: 9


In [4]:
# Cell 3: FiLM module (pure parent attraction, no repulsion)

class FiLMGen(nn.Module):
    """
    FiLM generator that maps (w1, w2) to a modulated vector z.

    z = gamma([w1, w2]) * base + beta([w1, w2]),
    where base = 0.5 * (w1 + w2).
    """
    def __init__(self, dim: int):
        super().__init__()
        hidden = 512
        self.gammanet = nn.Sequential(
            nn.Linear(dim * 2, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Linear(hidden, dim),
            nn.Sigmoid(),  # gating in [0,1]
        )
        self.betanet = nn.Sequential(
            nn.Linear(dim * 2, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Linear(hidden, dim),
            nn.Tanh(),     # shift in [-1,1]
        )

    def forward(self, w1: torch.Tensor, w2: torch.Tensor) -> torch.Tensor:
        """
        w1, w2: (..., D)
        returns z: (..., D)
        """
        context = torch.cat([w1, w2], dim=-1)
        gamma = self.gammanet(context)
        beta = self.betanet(context)
        base = 0.5 * (w1 + w2)
        z = gamma * base + beta
        return z


def cosine(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Cosine similarity along last dim."""
    return F.cosine_similarity(a, b, dim=-1)


In [5]:
# Cell 4: Build training tensors from triplets

def build_training_tensors(
    triplets: List[Tuple[str, str, str]],
    embs: Dict[str, np.ndarray],
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Build (w1, w2, wp) tensors from word-based triplets.
    """
    v1_list, v2_list, vp_list = [], [], []
    for w1, w2, wp in triplets:
        v1_list.append(embs[w1])
        v2_list.append(embs[w2])
        vp_list.append(embs[wp])
    v1 = torch.tensor(np.stack(v1_list), dtype=torch.float32, device=DEVICE)
    v2 = torch.tensor(np.stack(v2_list), dtype=torch.float32, device=DEVICE)
    vp = torch.tensor(np.stack(vp_list), dtype=torch.float32, device=DEVICE)
    return v1, v2, vp


train_v1, train_v2, train_vp = build_training_tensors(valid_triplets, embs)
print("Train tensors shape:", train_v1.shape, train_v2.shape, train_vp.shape)


Train tensors shape: torch.Size([9, 300]) torch.Size([9, 300]) torch.Size([9, 300])


In [6]:
# Cell 5: Parent-attraction loss

class ParentAttractionLoss(nn.Module):
    """
    Simple cosine-based attraction loss:
        loss = 1 - cos(pred, target)
    """
    def __init__(self):
        super().__init__()

    def forward(
        self,
        pred: torch.Tensor,
        target: torch.Tensor,
    ) -> torch.Tensor:
        """
        pred, target: (B, D)
        """
        cos_sim = cosine(pred, target)  # (B,)
        loss = 1.0 - cos_sim
        return loss.mean()


In [9]:
# Cell 6: Train FiLM on hand-crafted triplets

model = FiLMGen(EMB_DIM).to(DEVICE)
criterion = ParentAttractionLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 2000

print("Training FiLM (pure parent attraction)...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    pred = model(train_v1, train_v2)      # (B, D)
    loss = criterion(pred, train_vp)

    loss.backward()
    optimizer.step()

    if epoch % 200 == 0 or epoch == 1:
        with torch.no_grad():
            cos_before = cosine(0.5 * (train_v1 + train_v2), train_vp).mean().item()
            cos_after = cosine(pred, train_vp).mean().item()
        print(
            f"[Epoch {epoch:4d}] loss = {loss.item():.4f} | "
            f"avg cos(avg,parent) = {cos_before:.4f} | "
            f"avg cos(FiLM,parent) = {cos_after:.4f}"
        )


Training FiLM (pure parent attraction)...
[Epoch    1] loss = 0.6203 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 0.3797
[Epoch  200] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch  400] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch  600] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch  800] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch 1000] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch 1200] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch 1400] loss = 0.0001 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 0.9999
[Epoch 1600] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch 1800] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch 2000] loss = 0.0000 | avg cos(avg,parent) = 0.7532 

In [10]:
# Cell 7: Top-k neighbor search

# Pre-build vocab tensor for neighbor lookup
all_words = list(embs.keys())
all_vecs = torch.tensor(
    np.stack([embs[w] for w in all_words], axis=0),
    dtype=torch.float32,
    device=DEVICE,
)
all_norm = F.normalize(all_vecs, p=2, dim=1)  # (V, D)


def topk_neighbors(
    query: torch.Tensor,
    k: int = 10,
) -> List[Tuple[str, float]]:
    """
    Return top-k neighbors and cosine scores for a query vector.

    query: (D,) or (1, D)
    """
    if query.dim() == 1:
        q = query.unsqueeze(0)
    else:
        q = query
    q_norm = F.normalize(q, p=2, dim=1)       # (1, D)
    sims = torch.matmul(all_norm, q_norm.T).squeeze(1)  # (V,)
    vals, idxs = torch.topk(sims, k=k)
    results: List[Tuple[str, float]] = []
    for v, idx in zip(vals.tolist(), idxs.tolist()):
        results.append((all_words[idx], float(v)))
    return results


In [11]:
# Cell 8: Evaluation — avg vs FiLM output

def evaluate_triplet(
    w1: str,
    w2: str,
    wp: str,
    model: FiLMGen,
    k: int = 10,
) -> None:
    """
    Print cosine to parent and top-k neighbors for
    - average vector
    - FiLM output
    """
    v1 = get_vec(w1, embs).unsqueeze(0)
    v2 = get_vec(w2, embs).unsqueeze(0)
    vp = get_vec(wp, embs).unsqueeze(0)

    v_avg = 0.5 * (v1 + v2)
    with torch.no_grad():
        z = model(v1, v2)

    cos_avg_parent = cosine(v_avg, vp).item()
    cos_z_parent = cosine(z, vp).item()
    cos_avg_child1 = cosine(v_avg, v1).item()
    cos_avg_child2 = cosine(v_avg, v2).item()
    cos_z_child1 = cosine(z, v1).item()
    cos_z_child2 = cosine(z, v2).item()

    print("=" * 60)
    print(f"Triplet: ({w1}, {w2}) -> {wp}")
    print(f"cos(avg, parent)  = {cos_avg_parent:.4f}")
    print(f"cos(FiLM, parent) = {cos_z_parent:.4f}")
    print(f"cos(avg, child1)  = {cos_avg_child1:.4f}")
    print(f"cos(avg, child2)  = {cos_avg_child2:.4f}")
    print(f"cos(FiLM, child1) = {cos_z_child1:.4f}")
    print(f"cos(FiLM, child2) = {cos_z_child2:.4f}")

    print("\nTop-k neighbors (avg):")
    n_avg = topk_neighbors(v_avg.squeeze(0), k=k)
    for i, (w, s) in enumerate(n_avg):
        print(f"  [{i:2d}] {w:15s}  {s:.4f}")

    print("\nTop-k neighbors (FiLM):")
    n_z = topk_neighbors(z.squeeze(0), k=k)
    for i, (w, s) in enumerate(n_z):
        print(f"  [{i:2d}] {w:15s}  {s:.4f}")


print("=== Evaluation on valid triplets ===")
for w1, w2, wp in valid_triplets:
    evaluate_triplet(w1, w2, wp, model, k=10)


=== Evaluation on valid triplets ===
Triplet: (cat, dog) -> animal
cos(avg, parent)  = 0.8186
cos(FiLM, parent) = 1.0000
cos(avg, child1)  = 0.9694
cos(avg, child2)  = 0.9748
cos(FiLM, child1) = 0.7893
cos(FiLM, child2) = 0.8013

Top-k neighbors (avg):
  [ 0] dog              0.9748
  [ 1] cat              0.9694
  [ 2] dogs             0.9095
  [ 3] cats             0.9014
  [ 4] pet              0.8743
  [ 5] puppy            0.8657
  [ 6] kitten           0.8384
  [ 7] pup              0.8340
  [ 8] pets             0.8322
  [ 9] kitty            0.8256

Top-k neighbors (FiLM):
  [ 0] animal           1.0000
  [ 1] animals          0.9289
  [ 2] pet              0.8193
  [ 3] human            0.8037
  [ 4] dog              0.8013
  [ 5] pets             0.7965
  [ 6] cats             0.7910
  [ 7] cat              0.7893
  [ 8] dogs             0.7885
  [ 9] humans           0.7834
Triplet: (tiger, lion) -> animal
cos(avg, parent)  = 0.6950
cos(FiLM, parent) = 1.0000
cos(avg, child1

In [12]:
# Cell 9 (stub): Hook to fuzzy FCA extent seed

def embed_query_to_extent(
    query_vec: torch.Tensor,
    X: torch.Tensor,
    temperature: float = 0.1,
) -> torch.Tensor:
    """
    Convert a query vector into a soft fuzzy extent over object pool X.

    query_vec : (D,)
    X         : (B, D)
    returns   : extent (B,) in [0, 1]
    """
    if query_vec.dim() == 1:
        q = query_vec.unsqueeze(0)
    else:
        q = query_vec
    B = X.shape[0]
    sims = cosine(
        X,
        q.expand(B, -1),
    )  # (B,)
    extent = F.softmax(sims / temperature, dim=0)
    return extent
